<div class="alert alert-block alert-success">
<em>Panelite</em> is powered by young technologies like <a href="https://pyodide.org/en/stable/">Pyodide</a> and <a href="https://jupyterlite.readthedocs.io/en/latest/">Jupyterlite</a>. Some browsers may be poorly supported (e.g. mobile or 32-bit versions). If you experience issues, please <a href="https://github.com/holoviz/panel/issues">report them</a>.
</div>

# Accordion



In [ ]:
import panel as pn
import panel.ui

pn.extension()


The `Accordion` component creates expandable/collapsible sections. The labels for each section may be defined explicitly as part of a tuple, and otherwise default to the name parameter of the card’s contents. Like Column and Row, Tabs has a list-like API that allows interactively updating and modifying the cards using methods append, extend, clear, insert, pop, remove and `__setitem__` (to replace the contents).

## Parameters:

### Core

* **`active`** (`list[int]`): List of indexes of currently expanded sections.
* **`disabled`** (`list[int]`): List of indexes of disabled sections.
* **`dynamic`** (`boolean`): Whether to render section contents only when expanded.
* **`toggle`** (`boolean`): If True, only one section can be expanded at a time.

### Style

* **`active_header_background`** (`str`): The text color of the header when the Card is expanded.
* **`active_header_color`** (`str`): The background color of the header when the Card is expanded.
* **`disable_gutters`** (`boolean`): Whether to disable margins between expanded sections.
* **`elevation`** (`int`): The elevation level of the card surface.
* **`header_background`** (`str`): The background color of the header.
* **`header_color`** (`str`): The color of the header text.
* **`raised`** (`boolean`): Whether the card appears elevated above the background.
* **`square`** (`boolean`): Whether to disable rounded corners.
* **`title_variant`** (`str`): The text variant of the header title (default="h3").
* **`variant`** (`Literal["filled", "outlined"]`): Whether to show an outline instead of elevation.

---

### Basic Usage

The `Accordion` renders a list of items into a stack of collapsible cards. The labels can be provided as tuples and whether a card is expanded is controlled via the `active parameter:



In [ ]:
pn.ui.Accordion(
    ('Title 1', 'My content'),
    ('Title 2', 'My secondary content'),
    active=[1]
)


### Objects & Headers

An Accordion layout can either be instantiated as empty and populated after the fact, or by using a list of objects provided on instantiation as positional arguments.

If the objects are not already Panel components they will each be converted to one using the pn.panel conversion method. 

Unlike other panel types, Accordion also accepts tuples to specify the title of each tab; if no name is supplied explicitly the name of the underlying object will be used:



In [ ]:
from bokeh.plotting import figure

p1 = figure(width=300, height=300, name='Scatter', margin=5)
p1.scatter([0, 1, 2, 3, 4, 5, 6], [0, 1, 2, 3, 2, 1, 0])

p2 = figure(width=300, height=300, name='Line', margin=5)
p2.line([0, 1, 2, 3, 4, 5, 6], [0, 1, 2, 3, 2, 1, 0])

accordion = pn.ui.Accordion(('Scatter', p1), p2, margin=10)
accordion


The contents of the Accordion.objects list should never be modified individually, because Panel cannot detect when items in that list have changed internally, and will thus fail to update any already-rendered views of those objects (and their card titles!). Instead, use the provided methods for adding and removing items from the list. The only change that is safe to make directly to Accordion.objects is to replace the list of objects entirely. As a simple example of using the methods, we might add an additional widget to the Accordion using the append method:



In [ ]:
p3 = figure(width=300, height=300, name='Square', margin=5)
p3.scatter([0, 1, 2, 3, 4, 5, 6], [0, 1, 2, 3, 2, 1, 0], marker='square', size=10)

accordion.append(('Square', p3))


On a live server or in a notebook the accordion displayed above will dynamically expand to include the new card. To see the effect in a statically rendered page, we will display the accordion a second time:



In [ ]:
accordion


#### HTML & Component Titles

The titles of each card may also contain HTML or render Panel components:



In [ ]:
component_title = pn.ui.Row(
    pn.ui.IconButton(icon='show_chart', size='small', margin=(10, 0)),
    '#### Line'
)

line = figure(width=300, height=300, name='Line')
line.line([0, 1, 2, 3, 4, 5, 6], [0, 1, 2, 3, 2, 1, 0])

html_title = """
<p style="display: inline-flex; align-items: center;">
  <span class="material-icons" style="font-size: 1.2em; padding-right: 1em;">square</span>
  <span>Square</span>
</p>"""

square = figure(width=300, height=300, name='Square')
square.scatter([0, 1, 2, 3, 4, 5, 6], [0, 1, 2, 3, 2, 1, 0], marker='square', size=10)

pn.ui.Accordion(
    (component_title, line),
    (html_title, square)
)


### `active`

In addition to being able to modify the objects using methods we can also get and set the currently active cards as a list of integers, which will update any rendered views of the object.

The indexes in `active` refer to positions in `objects`, so they follow the sections when one is removed: `pop`, `remove` and `clear` drop the index of the removed section and shift the indexes of all later sections down by one. Assigning a shorter list to `objects` directly drops the indexes that are now out of range.

The same object may be listed in more than one section, but it can only be displayed in one place at a time. It is rendered in the first expanded section that lists it, and the other sections listing it appear empty:



In [ ]:
pn.ui.Accordion(p1, p2, p3, active=[0, 2], margin=10)


### `disabled`

The `disabled` parameter also contains indexes but instead controls which section can't be expanded. Like `active`, the indexes are shifted when a section is removed, so the remaining disabled sections stay disabled.



In [ ]:
pn.ui.Accordion(
    'Foo', 'Bar', 'Baz', disabled=[2]
)


### `toggle`

When toggle is enabled only one card can be active at the same time, i.e., expanding one card will collapse the other active cards (much like a Tabs layout).



In [ ]:
accordion.clone(toggle=True, active=[0])


### `dynamic`

When enabled the dynamic option ensures that only the expanded cards are actually rendered, the contents of a card are only loaded when it is expanded and are discarded again when it is collapsed. This can be very helpful in a server or notebook context when displaying a lot of cards or when the individual objects are large or expensive to render. Note however that without a live server the contents of collapsed cards will never load.

Independent of `dynamic`, the contents of a collapsed section are removed from the page and attached again when it is expanded, just like a collapsed `Card` or `Details`. State that only lives in the browser is therefore reset when a section collapses: an embedded iframe reloads, audio or video playback stops and the scroll position inside the section is lost. With `dynamic=False` the components themselves are kept, so state that is synced back to Python, such as widget values, is preserved. Anything that must keep running while hidden should be displayed outside the `Accordion`:



In [ ]:
accordion.clone(dynamic=True, active=[0])